# 05 — Split dataset ABSA

Jalankan notebook dari folder `repo_bank_news` menggunakan kernel Conda `DeepLearning`. Sumber data adalah `data/annotation/annotated_batch1.csv`. Label: 0 negatif, 1 netral, 2 positif. Split mendekati 70/15/15 menjaga distribusi kelas; artikel dan teks identik dikelompokkan untuk mencegah kebocoran.

## Buat split

Sel berikut menulis ulang `data/final/train.csv`, `val.csv`, `test.csv`, dan `distribution.json` secara deterministik menggunakan seed yang sama.

In [ ]:
"""Article-grouped, approximately stratified 70/15/15 split; no resampling."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd()
df = pd.read_csv(ROOT / 'data/annotation/annotated_batch1.csv', dtype={'id_artikel': str})
assert not df[['id_artikel', 'kalimat', 'nama_aspek', 'sentimen']].isna().any().any()
assert set(df.sentimen.unique()) == {0, 1, 2}
assert not df.id_kalimat.duplicated().any()
invalid = df.aspek_salah.fillna('').astype(str).str.lower().str.strip().isin(['x', 'ya', '1', 'true'])
df = df.loc[~invalid].reset_index(drop=True)

# Group any articles sharing the same normalized sentence; this prevents exact
# duplicate text from leaking across partitions, in addition to article leakage.
parent = {a: a for a in df.id_artikel}
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[b] = a
for _, g in df.assign(_text=df.kalimat.str.strip().str.casefold()).groupby('_text'):
    ids = g.id_artikel.unique()
    for a in ids[1:]: union(ids[0], a)
groups = df.id_artikel.map(find).to_numpy()

def pick(indices, frac, seed):
    sub = df.iloc[indices]
    target = sub.sentimen.value_counts(normalize=True).reindex([0,1,2], fill_value=0).to_numpy()
    candidates = GroupShuffleSplit(n_splits=600, test_size=frac, random_state=seed)
    best = None
    for train, hold in candidates.split(sub, groups=groups[indices]):
        counts = sub.iloc[hold].sentimen.value_counts(normalize=True).reindex([0,1,2], fill_value=0).to_numpy()
        score = abs(len(hold)/len(sub)-frac) + np.abs(counts-target).sum()
        if best is None or score < best[0]: best = (score, indices[train], indices[hold])
    return best[1], best[2]

trainval, test = pick(np.arange(len(df)), .15, 42)
train, val = pick(trainval, .15/.85, 43)
parts = {'train': df.iloc[train], 'val': df.iloc[val], 'test': df.iloc[test]}
out = ROOT/'data/final'; out.mkdir(parents=True, exist_ok=True)
for name, part in parts.items(): part.to_csv(out/f'{name}.csv', index=False)
for a in parts:
    for b in parts:
        if a >= b: continue
        assert set(parts[a].id_artikel).isdisjoint(parts[b].id_artikel)
        assert set(parts[a].kalimat.str.strip().str.casefold()).isdisjoint(parts[b].kalimat.str.strip().str.casefold())
summary = {name: {'rows':len(p), 'articles':int(p.id_artikel.nunique()), 'classes':{str(k):int(v) for k,v in p.sentimen.value_counts().sort_index().items()}} for name,p in parts.items()}
(out/'distribution.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


## Periksa distribusi dan kebocoran

In [ ]:
import pandas as pd
from pathlib import Path
parts = {s: pd.read_csv(Path('data/final') / f'{s}.csv', dtype={'id_artikel': str}) for s in ['train', 'val', 'test']}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert set(parts[a].id_artikel).isdisjoint(parts[b].id_artikel)
    assert set(parts[a].kalimat.str.strip().str.casefold()).isdisjoint(parts[b].kalimat.str.strip().str.casefold())
pd.DataFrame({s: p.sentimen.value_counts().reindex([0, 1, 2], fill_value=0) for s, p in parts.items()}).rename(index={0:'negatif', 1:'netral', 2:'positif'})